# 02 - Baseline Model Training and Evaluation

This notebook establishes a baseline for our match outcome and scoreline prediction models. It will demonstrate how to load data, generate features, train the models, and perform a basic evaluation.

In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join("..")))

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, mean_squared_error
from src.scraper import get_past_match_results
from src.features import generate_features
from src.model import MatchOutcomeModel, ScorelineModel, save_model
import os

# Define paths
MODELS_PATH = os.path.abspath(os.path.join("..", "models"))
os.makedirs(MODELS_PATH, exist_ok=True)

# Load historical match results (using dummy data for now)
past_results_df = get_past_match_results(start_date='2022-01-01', end_date='2026-06-15')
past_results_df['date'] = pd.to_datetime(past_results_df['date'])

print("First 5 rows of past results:")
print(past_results_df.head())

[Scraper] Gathering integrated past match results...
[Scraper] Downloading rich historical data from Open GitHub Data source...
[Scraper] Fetching finished World Cup matches from live API...
[Scraper] Added 19 finished matches from live API.
[Scraper] Total dataset compiled successfully: 19 matches.
First 5 rows of past results:
        match_id       date      home_team           away_team  home_score  \
0  M_real_537327 2026-06-11         Mexico        South Africa           2   
1  M_real_537328 2026-06-12    South Korea             Czechia           2   
2  M_real_537333 2026-06-12         Canada  Bosnia-Herzegovina           1   
3  M_real_537345 2026-06-13  United States            Paraguay           4   
4  M_real_537334 2026-06-13          Qatar         Switzerland           1   

   away_score  competition_id  
0           0            2000  
1           1            2000  
2           1            2000  
3           1            2000  
4           1            2000  


In [2]:
# Generate features
# For baseline, we use past_results_df as both the matches to generate features for and the historical data
features_df = generate_features(past_results_df, past_results_df)

# Prepare data for Match Outcome Model
# Map actual results to numerical labels: 0=Away Win, 1=Draw, 2=Home Win
features_df["actual_outcome_label"] = features_df.apply(lambda row: 2 if row["home_score"] > row["away_score"] else (0 if row["home_score"] < row["away_score"] else 1), axis=1)

# Initialize model first to safely extract required features
outcome_model = MatchOutcomeModel()
X = features_df[outcome_model.features]
y = features_df["actual_outcome_label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train Match Outcome Model
outcome_model.train(X_train, y_train)
save_model(outcome_model, os.path.join(MODELS_PATH, "match_outcome_model.joblib"))

# Evaluate Match Outcome Model
y_pred_proba = outcome_model.predict_proba(X_test)
baseline_log_loss = log_loss(y_test, y_pred_proba, labels=[0, 1, 2])
print(f"Baseline Match Outcome Log-Loss: {baseline_log_loss:.4f}")

[Features] Generating Elo ratings...
[Features] Generating rolling form...
[Features] Generated features for 19 matches.
[Model] Training Match Outcome Model...
[Model] Match Outcome Model trained.
[Model] Model saved to C:\Users\ASUS A412\DS\World Cup 2026\models\match_outcome_model.joblib
Baseline Match Outcome Log-Loss: 0.0000


In [3]:
# Prepare data for Scoreline Model
home_goals = features_df["home_score"]
away_goals = features_df["away_score"]

home_goals_train, home_goals_test = train_test_split(home_goals, test_size=0.2, random_state=42)
away_goals_train, away_goals_test = train_test_split(away_goals, test_size=0.2, random_state=42)

# Train Scoreline Model
score_model = ScorelineModel()
score_model.train(X_train, home_goals_train, away_goals_train)
save_model(score_model, os.path.join(MODELS_PATH, "scoreline_model.joblib"))

# Evaluate Scoreline Model (simplified RMSE)
baseline_rmse = score_model.evaluate(X_test, home_goals_test, away_goals_test)
print(f"Baseline Scoreline RMSE: {baseline_rmse:.4f}")

[Model] Training Scoreline Model (simplified Poisson)...
[Model] Updated average home goals: 1.93, away goals: 1.07
[Model] Model saved to C:\Users\ASUS A412\DS\World Cup 2026\models\scoreline_model.joblib
Baseline Scoreline RMSE: 1.0000
